# 02 — Evaluation

Evaluate the trained model on the held-out test set.

**Covers**: AUC-ROC per class, precision-recall curves, confusion matrix, comparison with CheXNet benchmarks.

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from src.utils.config import PATHOLOGY_CLASSES, Paths
print('Setup OK')

## 2. Run Evaluation

In [ ]:
from src.training.evaluate import evaluate_on_test_set

metrics = evaluate_on_test_set(save_plots=True)
print(f'Mean AUC  : {metrics.mean_auc:.4f}')
print(f'Mean AUPRC: {metrics.mean_auprc:.4f}')

## 3. Per-class AUC-ROC vs CheXNet Benchmark

In [ ]:
# Published CheXNet AUC scores for comparison
cheXnet_auc = {
    'Atelectasis': 0.8094, 'Cardiomegaly': 0.9248,
    'Consolidation': 0.7901, 'Edema': 0.8878,
    'Effusion': 0.8638, 'Mass': 0.8676,
    'Nodule': 0.7802, 'Pleural_Thickening': 0.8062,
    'Pneumonia': 0.7680, 'Pneumothorax': 0.8887,
}

fig = go.Figure()
fig.add_trace(go.Bar(
    name='Our model', x=PATHOLOGY_CLASSES,
    y=[metrics.auc_per_class.get(c, 0) for c in PATHOLOGY_CLASSES],
    marker_color='steelblue'
))
fig.add_trace(go.Bar(
    name='CheXNet (2017)', x=PATHOLOGY_CLASSES,
    y=[cheXnet_auc.get(c, 0) for c in PATHOLOGY_CLASSES],
    marker_color='lightcoral'
))
fig.update_layout(
    barmode='group', title='AUC-ROC: Our Model vs CheXNet',
    yaxis=dict(range=[0, 1], title='AUC-ROC'),
    height=400, legend=dict(orientation='h')
)
fig.show()

## 4. ROC Curves

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

roc_path = Paths.processed / 'roc_curves_test.png'
if roc_path.exists():
    img = Image.open(roc_path)
    plt.figure(figsize=(16, 7))
    plt.imshow(img)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('Run evaluate_on_test_set(save_plots=True) first')

## 5. Summary Table

In [ ]:
from src.models.metrics import metrics_to_dataframe
df = metrics_to_dataframe(metrics)
df.round(4).style.highlight_max(subset=['auc', 'f1'], color='lightgreen')